In [99]:
import gspread
import google.auth

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import sqlite3
import os

from dotenv import load_dotenv

In [100]:
df = pd.read_csv("./tally-subs/Tá Na Mesa - Pesquisa de Campo - Página1.csv")
#df = pd.DataFrame(planilha)
#df = df.rename(columns=df.iloc[0]).drop(df.index[0])

df_columns = list(df.columns)
df_columns[0] = "Submission ID"
df_columns[1] = "Respondent ID"
df_columns[2] = "Submitted at"

df.columns = df_columns

In [101]:
df.columns

Index(['Submission ID', 'Respondent ID', 'Submitted at',
       'Selecione o seu município', 'Unnamed: 4',
       'Você é inscrito no programa CadÚnico?',
       'Qual é a sua condição de trabalho atual?',
       'Quantas pessoas moram com você?',
       'Nos últimos meses, você teve medo da comida da sua casa acabar antes de ter dinheiro para comprar mais?',
       'Qual é a sua renda familiar mensal?',
       'No restaurante do programa Tá Na Mesa, você recebe apenas a sua refeição?',
       'As refeições do programa Tá Na Mesa servem todas as pessoas da sua casa?',
       'Nos últimos meses, você sentiu fome e não comeu por falta de dinheiro?',
       'Contando com a refeição distribuída pelo programa Tá Na Mesa, quantas refeições você faz por dia?',
       'Sem o programa Tá Na Mesa, você continua se alimentando normalmente?',
       'Com que frequência você consegue receber alguma refeição do programa Tá Na Mesa?',
       'Em relação à limpeza do restaurante, qual a sua opinião?',

In [102]:
# Remove colunas vazias de verdade: valores em branco, espaços ou totalmente nulos.
df.replace(r'^\s*$', np.nan, regex=True, inplace=True)
df = df.dropna(how="all", axis=1)
df = df.loc[:, df.columns.astype(str).str.strip() != ""]

df['Indice manual'] = range(1, len(df) + 1)

len(df.columns)

46

In [103]:
df = df.drop(columns=[col for col in df.columns if col == ""], errors='ignore')

# Ou, se quiser remover colunas que são "Unnamed" (comum no Pandas)
df = df.loc[:, ~df.columns.str.contains('^Unnamed', na=False)]
df

,Submission ID,Respondent ID,Submitted at,Selecione o seu município,Você é inscrito no programa CadÚnico?,Qual é a sua condição de trabalho atual?,Quantas pessoas moram com você?,"Nos últimos meses, você teve medo da comida da sua casa acabar antes de ter dinheiro para comprar mais?",Qual é a sua renda familiar mensal?,"No restaurante do programa Tá Na Mesa, você recebe apenas a sua refeição?",...,Você já deixou de comer alguma refeição do programa Tá Na Mesa por que não gostou do cardápio do dia?,Em que momento você descobre o cardápio que será servido no dia?,"Quando você não gosta do cardápio do dia, o que você faz com a refeição?",A sua sugestão de mudança do cardápio foi atendida?,"Para uma única pessoa, a quantidade de comida das refeições é suficiente?","A quantidade de carne, frango ou peixe servida nas refeições é suficiente?","Levando em conta todas as refeições que você já recebeu durante o programa Tá Na Mesa, como você avalia o sabor das refeições?","Em relação a todo o programa Tá Na Mesa, qual o seu nível de satisfação?","Na sua opinião, o programa Tá Na Mesa precisa ser continuado?",Indice manual
0,PRDkx7x,MedkPJM,2026-04-17 11:37:52,João Pessoa,"Sim, sou inscrito",Desempregado(a),Moro com 3(três) pessoas ou mais,"Sim, tive medo da comida acabar",Até 1(um) salário mínimo,"Não, recebo a minha refeição e a de mais uma p...",...,"Nunca, pois como a refeição mesmo se não gosta...",Apenas após receber a refeição ou comprar a ficha,Como a refeição mesmo assim,"Não, não foi atendida","Sim, é suficiente pois fico saciado(a) com a q...","Sim, é suficiente pois a quantidade de carne, ...",Bom,Satisfeito,"Sim, precisa ser continuado",1
1,5X4MNVv,Ekbjg1A,2026-04-17 11:44:16,João Pessoa,"Sim, sou inscrito",Desempregado(a),Moro sozinho(a),"Não, não tive medo da comida acabar",Até 1(um) salário mínimo,"Sim, recebo apenas a minha refeição",...,Raramente deixo de comer alguma refeição do pr...,Apenas após receber a refeição ou comprar a ficha,Como a refeição mesmo assim,NaN,"Sim, é suficiente pois fico saciado(a) com a q...","Sim, é suficiente pois a quantidade de carne, ...",Bom,Satisfeito,"Sim, precisa ser continuado",2
2,rDBYy5L,7R9Bz7Z,2026-04-17 12:01:25,Curral de Cima,"Não, não sou inscrito",Desempregado(a),Moro com 2(duas) pessoas,"Não, não tive medo da comida acabar",Até 1(um) salário mínimo,"Não, recebo a minha refeição e a de mais uma p...",...,"Nunca, gosto de todas as refeições do cardápio...",Apenas após receber a refeição ou comprar a ficha,Como a refeição mesmo assim,"Sim, foi atendida","Sim, é suficiente pois fico saciado(a) com a q...","Sim, é suficiente pois a quantidade de carne, ...",Muito bom,Satisfeito,"Sim, precisa ser continuado",3
3,zENd5ZZ,aQjN54v,2026-04-17 13:58:38,Sobrado,"Sim, sou inscrito",Trabalho informalmente,Moro com 3(três) pessoas ou mais,"Sim, tive medo da comida acabar",Até 1(um) salário mínimo,"Sim, recebo apenas a minha refeição",...,"Nunca, gosto de todas as refeições do cardápio...",Antes de chegar no restaurante,Como a refeição mesmo assim,NaN,"Sim, é suficiente pois fico saciado(a) com a q...","Sim, é suficiente pois a quantidade de carne, ...",Muito bom,Muito satisfeito,"Não, não deve ser continuado",4
4,o9D4xrP,9q6WoMp,2026-04-20 13:28:13,João Pessoa,"Sim, sou inscrito",Trabalho informalmente,Moro com 2(duas) pessoas,"Não, não tive medo da comida acabar",Até 1(um) salário mínimo,"Não, recebo a minha refeição e a de mais uma p...",...,"Nunca, gosto de todas as refeições do cardápio...",Antes de comprar a ficha e já no restaurante,Como a refeição mesmo assim,"Não, não foi atendida","Sim, é suficiente pois fico saciado(a) com a q...","Sim, é suficiente pois a quantidade de carne, ...",Bom,Satisfeito,"Sim, precisa ser continuado",5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
974,5Xk7xvQ,RGkBbxp,2026-06-09 17:33:41,Olho d’Água,"Sim, sou inscrito",Desempregado(a),Moro com 1(uma) pessoa,"Sim, tive medo da comida acabar",Até 1(um) salário mínimo,

In [104]:
def get_dia(x):
    return (x - data_primeira_sub).days + 1

def get_semana(x):
    return ((x - data_primeira_sub).days // 7 ) + 1


df['Submitted at'] = pd.to_datetime(df['Submitted at'])


data_primeira_sub = df['Submitted at'].min()

# Criação da coluna Submitted_at_dia
df['Submitted at_dia'] = df['Submitted at'].apply(lambda x : get_dia(x))

# Criação da coluna Submitted_at_semana
df['Submitted at_semana'] = df['Submitted at'].apply(lambda x : get_semana(x))

In [105]:
df.columns

Index(['Submission ID', 'Respondent ID', 'Submitted at',
       'Selecione o seu município', 'Você é inscrito no programa CadÚnico?',
       'Qual é a sua condição de trabalho atual?',
       'Quantas pessoas moram com você?',
       'Nos últimos meses, você teve medo da comida da sua casa acabar antes de ter dinheiro para comprar mais?',
       'Qual é a sua renda familiar mensal?',
       'No restaurante do programa Tá Na Mesa, você recebe apenas a sua refeição?',
       'As refeições do programa Tá Na Mesa servem todas as pessoas da sua casa?',
       'Nos últimos meses, você sentiu fome e não comeu por falta de dinheiro?',
       'Contando com a refeição distribuída pelo programa Tá Na Mesa, quantas refeições você faz por dia?',
       'Sem o programa Tá Na Mesa, você continua se alimentando normalmente?',
       'Com que frequência você consegue receber alguma refeição do programa Tá Na Mesa?',
       'Em relação à limpeza do restaurante, qual a sua opinião?',
       'Você sabe qu

In [106]:
# Criação do KPI_1 V.A.G.
opcoes_q6 = [
    'Sim, deixei de comer por falta de dinheiro',
    'Sim, frequentemente deixo de fazer alguma refeição por falta de dinheiro'
]

condicao_q5 = df['Nos últimos meses, você teve medo da comida da sua casa acabar antes de ter dinheiro para comprar mais?'] == 'Sim, tive medo da comida acabar'
condicao_q6 = df['Nos últimos meses, você sentiu fome e não comeu por falta de dinheiro?'].isin(opcoes_q6)

df['kpi_vag'] = np.where(condicao_q5 & condicao_q6, 1, 0)

In [107]:
# Criação do KPI_7 Dependência dos beneficiários em relação ao programa

opcoes_q10 = [
    'Faço apenas a refeição servida pelo programa', 'Faço apenas 2(duas) refeições por dia, contando com a refeição servida pelo programa'
]
condicao_q10 = df['Contando com a refeição distribuída pelo programa Tá Na Mesa, quantas refeições você faz por dia?'].isin(opcoes_q10)
condicao_q9 = df['Sem o programa Tá Na Mesa, você continua se alimentando normalmente?'] == 'Não, dependo totalmente da refeição servida pelo programa'


df['kpi_dependencia'] = np.where(condicao_q9 & condicao_q10, 1, 0)

In [108]:
def get_peso(x):
    if x == 'De 4(quatro) à 5(cinco) vezes por semana': return 4.5
    elif x == 'De 2(duas) à 3(três) vezes por semana': return 2.5
    elif x == 'Cerca de 1(uma) vez por semana': return 1
    elif x == 'Quinzenalmente': return 0.5

# preparação do dataframe para cálculo dos KPIs 2 e 3, que, na verdade, ambos são valores numéricos

df['peso_frequencia_semanal'] = df['Com que frequência você consegue receber alguma refeição do programa Tá Na Mesa?'].apply(lambda x : get_peso(x))
df

,Submission ID,Respondent ID,Submitted at,Selecione o seu município,Você é inscrito no programa CadÚnico?,Qual é a sua condição de trabalho atual?,Quantas pessoas moram com você?,"Nos últimos meses, você teve medo da comida da sua casa acabar antes de ter dinheiro para comprar mais?",Qual é a sua renda familiar mensal?,"No restaurante do programa Tá Na Mesa, você recebe apenas a sua refeição?",...,"A quantidade de carne, frango ou peixe servida nas refeições é suficiente?","Levando em conta todas as refeições que você já recebeu durante o programa Tá Na Mesa, como você avalia o sabor das refeições?","Em relação a todo o programa Tá Na Mesa, qual o seu nível de satisfação?","Na sua opinião, o programa Tá Na Mesa precisa ser continuado?",Indice manual,Submitted at_dia,Submitted at_semana,kpi_vag,kpi_dependencia,peso_frequencia_semanal
0,PRDkx7x,MedkPJM,2026-04-17 11:37:52,João Pessoa,"Sim, sou inscrito",Desempregado(a),Moro com 3(três) pessoas ou mais,"Sim, tive medo da comida acabar",Até 1(um) salário mínimo,"Não, recebo a minha refeição e a de mais uma p...",...,"Sim, é suficiente pois a quantidade de carne, ...",Bom,Satisfeito,"Sim, precisa ser continuado",1,1,1,0,0,4.5
1,5X4MNVv,Ekbjg1A,2026-04-17 11:44:16,João Pessoa,"Sim, sou inscrito",Desempregado(a),Moro sozinho(a),"Não, não tive medo da comida acabar",Até 1(um) salário mínimo,"Sim, recebo apenas a minha refeição",...,"Sim, é suficiente pois a quantidade de carne, ...",Bom,Satisfeito,"Sim, precisa ser continuado",2,1,1,0,0,4.5
2,rDBYy5L,7R9Bz7Z,2026-04-17 12:01:25,Curral de Cima,"Não, não sou inscrito",Desempregado(a),Moro com 2(duas) pessoas,"Não, não tive medo da comida acabar",Até 1(um) salário mínimo,"Não, recebo a minha refeição e a de mais uma p...",...,"Sim, é suficiente pois a quantidade de carne, ...",Muito bom,Satisfeito,"Sim, precisa ser continuado",3,1,1,0,0,4.5
3,zENd5ZZ,aQjN54v,2026-04-17 13:58:38,Sobrado,"Sim, sou inscrito",Trabalho informalmente,Moro com 3(três) pessoas ou mais,"Sim, tive medo da comida acabar",Até 1(um) salário mínimo,"Sim, recebo apenas a minha refeição",...,"Sim, é suficiente pois a quantidade de carne, ...",Muito bom,Muito satisfeito,"Não, não deve ser continuado",4,1,1,0,0,4.5
4,o9D4xrP,9q6WoMp,2026-04-20 13:28:13,João Pessoa,"Sim, sou inscrito",Trabalho informalmente,Moro com 2(duas) pessoas,"Não, não tive medo da comida acabar",Até 1(um) salário mínimo,"Não, recebo a minha refeição e a de mais uma p...",...,"Sim, é suficiente pois a quantidade de carne, ...",Bom,Satisfeito,"Sim, precisa ser continuado",5,4,1,0,1,4.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
974,5Xk7xvQ,RGkBbxp,2026-06-09 17:33:41,Olho d’Água,"Sim, sou inscrito",Desempregado(a),Moro com 1(uma) pessoa,"Sim, tive medo da comida acabar",Até 1(um) salário mínimo,"Não, recebo a minha refeição e a de mais uma p...",...,"Sim, é suficiente pois a quantidade de carne, ...",Muito bom,Muito satisfeito,"Sim, precisa ser continuado",975,54,8,0,0,4.5
975,vXBZ4pg,PdkBVB5,2026-06-09 17:59:17,Patos,"Sim, sou inscrito",Desempregado(a),Moro com 3(três) pessoas ou mais,"Sim, tive medo da comida acabar",Até 1(um) salário mínimo,"Não, recebo a minha refeição e a de mais uma p...",...,"Não, não é suficiente pois a quantidade de car...",Ruim,Satisfeito,"Sim, precisa ser continuado",976,54,8,0,0,2.5
976,kbAoO1d,9qM6GeG,2026-06-09 18:21:44,Montadas,"Sim, sou inscrito",Desempregado(a),Moro com 1(uma) pessoa,"Não, não tive medo da comida acabar",Até 1(um) salário mínimo,"Não, recebo a minha refeição e a de mais uma p...",...,"Sim, é suficiente pois a quantidade de carne, ...",Muito bom,Muito satisfeito,"Sim, precisa ser continuado",977,54,8,0,0,4.5
977,Ge1G5Pk,9qM6GeG,2026-06-09 18:38:27,Montadas,"Sim, sou inscrito",Trabalho informalmente,Moro com 3(três) pessoas ou mais,"Não, não tive medo da comida acabar",Até 1(um) salário mínimo,"Não, recebo a minha refeição e a de mais uma p...",...,"Sim, é suficiente pois a quantidade de carne, ...",Muito bom

In [109]:
def get_peso_tempo_espera(x):
    if x == "Não preciso esperar, recebo a refeição rapidamente": return 0.25
    elif x == "Espero até 30(trinta) minutos": return 0.5
    elif x == "Espero de 30(trinta) minutos à 1(uma) hora": return 0.75
    elif x == "Espero de 1(uma) hora à 2(duas) horas": return 1.5
    elif x == "Espero por mais de 2(duas) horas": return 2.5


df['peso_tempo_espera'] = df['Quanto tempo você espera na fila para receber a sua refeição?'].apply(lambda x : get_peso_tempo_espera(x))
df['peso_tempo_espera']

0      0.75
1      0.75
2      0.25
3       NaN
4      2.50
       ... 
974     NaN
975     NaN
976     NaN
977     NaN
978     NaN
Name: peso_tempo_espera, Length: 979, dtype: float64

In [110]:
# preparação para cálculo dos KPIs 4 e 4.1, que, na verdade, ambos são valores numéricos

df['ja_ficou_sem_receber_refeicao_mesmo_na_fila'] = df['Você já esperou na fila e mesmo assim ficou sem receber a refeição?'].apply(lambda x :
    0 if (x == 'Nunca, sempre que entrei na fila recebi a refeição' or x == 'Nunca, até hoje, sempre que entrei na fila recebi a refeição') else 1)
df

,Submission ID,Respondent ID,Submitted at,Selecione o seu município,Você é inscrito no programa CadÚnico?,Qual é a sua condição de trabalho atual?,Quantas pessoas moram com você?,"Nos últimos meses, você teve medo da comida da sua casa acabar antes de ter dinheiro para comprar mais?",Qual é a sua renda familiar mensal?,"No restaurante do programa Tá Na Mesa, você recebe apenas a sua refeição?",...,"Em relação a todo o programa Tá Na Mesa, qual o seu nível de satisfação?","Na sua opinião, o programa Tá Na Mesa precisa ser continuado?",Indice manual,Submitted at_dia,Submitted at_semana,kpi_vag,kpi_dependencia,peso_frequencia_semanal,peso_tempo_espera,ja_ficou_sem_receber_refeicao_mesmo_na_fila
0,PRDkx7x,MedkPJM,2026-04-17 11:37:52,João Pessoa,"Sim, sou inscrito",Desempregado(a),Moro com 3(três) pessoas ou mais,"Sim, tive medo da comida acabar",Até 1(um) salário mínimo,"Não, recebo a minha refeição e a de mais uma p...",...,Satisfeito,"Sim, precisa ser continuado",1,1,1,0,0,4.5,0.75,0
1,5X4MNVv,Ekbjg1A,2026-04-17 11:44:16,João Pessoa,"Sim, sou inscrito",Desempregado(a),Moro sozinho(a),"Não, não tive medo da comida acabar",Até 1(um) salário mínimo,"Sim, recebo apenas a minha refeição",...,Satisfeito,"Sim, precisa ser continuado",2,1,1,0,0,4.5,0.75,0
2,rDBYy5L,7R9Bz7Z,2026-04-17 12:01:25,Curral de Cima,"Não, não sou inscrito",Desempregado(a),Moro com 2(duas) pessoas,"Não, não tive medo da comida acabar",Até 1(um) salário mínimo,"Não, recebo a minha refeição e a de mais uma p...",...,Satisfeito,"Sim, precisa ser continuado",3,1,1,0,0,4.5,0.25,1
3,zENd5ZZ,aQjN54v,2026-04-17 13:58:38,Sobrado,"Sim, sou inscrito",Trabalho informalmente,Moro com 3(três) pessoas ou mais,"Sim, tive medo da comida acabar",Até 1(um) salário mínimo,"Sim, recebo apenas a minha refeição",...,Muito satisfeito,"Não, não deve ser continuado",4,1,1,0,0,4.5,NaN,1
4,o9D4xrP,9q6WoMp,2026-04-20 13:28:13,João Pessoa,"Sim, sou inscrito",Trabalho informalmente,Moro com 2(duas) pessoas,"Não, não tive medo da comida acabar",Até 1(um) salário mínimo,"Não, recebo a minha refeição e a de mais uma p...",...,Satisfeito,"Sim, precisa ser continuado",5,4,1,0,1,4.5,2.50,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
974,5Xk7xvQ,RGkBbxp,2026-06-09 17:33:41,Olho d’Água,"Sim, sou inscrito",Desempregado(a),Moro com 1(uma) pessoa,"Sim, tive medo da comida acabar",Até 1(um) salário mínimo,"Não, recebo a minha refeição e a de mais uma p...",...,Muito satisfeito,"Sim, precisa ser continuado",975,54,8,0,0,4.5,NaN,0
975,vXBZ4pg,PdkBVB5,2026-06-09 17:59:17,Patos,"Sim, sou inscrito",Desempregado(a),Moro com 3(três) pessoas ou mais,"Sim, tive medo da comida acabar",Até 1(um) salário mínimo,"Não, recebo a minha refeição e a de mais uma p...",...,Satisfeito,"Sim, precisa ser continuado",976,54,8,0,0,2.5,NaN,0
976,kbAoO1d,9qM6GeG,2026-06-09 18:21:44,Montadas,"Sim, sou inscrito",Desempregado(a),Moro com 1(uma) pessoa,"Não, não tive medo da comida acabar",Até 1(um) salário mínimo,"Não, recebo a minha refeição e a de mais uma p...",...,Muito satisfeito,"Sim, precisa ser continuado",977,54,8,0,0,4.5,NaN,1
977,Ge1G5Pk,9qM6GeG,2026-06-09 18:38:27,Montadas,"Sim, sou inscrito",Trabalho informalmente,Moro com 3(três) pessoas ou mais,"Não, não tive medo da comida acabar",Até 1(um) salário mínimo,"Não, recebo a minha refeição e a de mais uma p...",...,Muito satisfeito,"Sim, precisa ser continuado",978,54,8,0,1,4.5,NaN,1


In [111]:
df['Total diário de submissões'] = df.groupby('Submitted at_dia')['Submitted at_dia'].transform('count')

totais_diarios = df.groupby('Submitted at_dia')['Total diário de submissões'].first()
total_acumulado = totais_diarios.cumsum()

df['Total acumulado de submissões por dia'] = df['Submitted at_dia'].map(total_acumulado)

df['Média diária de submissões'] = df['Total acumulado de submissões por dia'] // df['Submitted at_dia']
df[['Total diário de submissões', 'Total acumulado de submissões por dia', 'Submitted at_dia', 'Média diária de submissões']]

,Total diário de submissões,Total acumulado de submissões por dia,Submitted at_dia,Média diária de submissões
0,4,4,1,4
1,4,4,1,4
2,4,4,1,4
3,4,4,1,4
4,8,12,4,3
...,...,...,...,...
974,178,979,54,18
975,178,979,54,18
976,178,979,54,18
977,178,979,54,18


In [112]:
df['Total semanal de submissões'] = df.groupby('Submitted at_semana')['Submitted at_semana'].transform('count')

totais_semanais= df.groupby('Submitted at_semana')['Total semanal de submissões'].first()
total_acumulado = totais_semanais.cumsum()

df['Total acumulado de submissões por semana'] = df['Submitted at_semana'].map(total_acumulado)

df['Média semanal de submissões'] = df['Total acumulado de submissões por semana'] // df['Submitted at_semana']
df[['Total semanal de submissões', 'Total acumulado de submissões por semana', 'Submitted at_semana', 'Média semanal de submissões']]

,Total semanal de submissões,Total acumulado de submissões por semana,Submitted at_semana,Média semanal de submissões
0,194,194,1,194
1,194,194,1,194
2,194,194,1,194
3,194,194,1,194
4,194,194,1,194
...,...,...,...,...
974,638,979,8,122
975,638,979,8,122
976,638,979,8,122
977,638,979,8,122


In [113]:
df["Total de beneficiários"] = len(df)*1

In [114]:
df.replace("", float("NaN"), inplace=True)
df.dropna(how="all", axis=1, inplace=True)

In [115]:
df[df["Submission ID"] == "5BMb1yQ"]

,Submission ID,Respondent ID,Submitted at,Selecione o seu município,Você é inscrito no programa CadÚnico?,Qual é a sua condição de trabalho atual?,Quantas pessoas moram com você?,"Nos últimos meses, você teve medo da comida da sua casa acabar antes de ter dinheiro para comprar mais?",Qual é a sua renda familiar mensal?,"No restaurante do programa Tá Na Mesa, você recebe apenas a sua refeição?",...,peso_frequencia_semanal,peso_tempo_espera,ja_ficou_sem_receber_refeicao_mesmo_na_fila,Total diário de submissões,Total acumulado de submissões por dia,Média diária de submissões,Total semanal de submissões,Total acumulado de submissões por semana,Média semanal de submissões,Total de beneficiários


In [116]:
print(len(df)) # cerca de 6 unidades

979


In [117]:
df.columns

Index(['Submission ID', 'Respondent ID', 'Submitted at',
       'Selecione o seu município', 'Você é inscrito no programa CadÚnico?',
       'Qual é a sua condição de trabalho atual?',
       'Quantas pessoas moram com você?',
       'Nos últimos meses, você teve medo da comida da sua casa acabar antes de ter dinheiro para comprar mais?',
       'Qual é a sua renda familiar mensal?',
       'No restaurante do programa Tá Na Mesa, você recebe apenas a sua refeição?',
       'As refeições do programa Tá Na Mesa servem todas as pessoas da sua casa?',
       'Nos últimos meses, você sentiu fome e não comeu por falta de dinheiro?',
       'Contando com a refeição distribuída pelo programa Tá Na Mesa, quantas refeições você faz por dia?',
       'Sem o programa Tá Na Mesa, você continua se alimentando normalmente?',
       'Com que frequência você consegue receber alguma refeição do programa Tá Na Mesa?',
       'Em relação à limpeza do restaurante, qual a sua opinião?',
       'Você sabe qu

In [118]:
len(df.columns)

60

In [119]:
import pandas as pd
import sqlite3
from sqlalchemy import create_engine
import os

string_conexao = os.getenv("RETOOL_CREDENTIALS")
engine_postgres = create_engine(string_conexao)

# 5. Enviar o DataFrame inteiro para o PostgreSQL
print("Enviando dados para o PostgreSQL no Retool...")

# O parâmetro if_exists='replace' cria a tabela do zero ou substitui a existente.
# Você também pode usar 'append' para apenas adicionar novas linhas.
df.to_sql(
    name='main', 
    con=engine_postgres, 
    if_exists='replace', 
    index=False
)

Enviando dados para o PostgreSQL no Retool...


434